In [1]:
!pip install thefuzz[speedup]
import numpy as np
from thefuzz import process
import pandas as pd

df=pd.read_csv("Raw_Data_24\Fantasy_season_2024_data.csv")
Xmins_data=pd.read_csv("ML_training2.csv").iloc[:,1:]
df=df[["Full_Name", "value", "kickoff_time","team_code","news","selected"]]
df['kickoff_time'] = pd.to_datetime(df['kickoff_time'])
result = df.loc[df.groupby('Full_Name')['kickoff_time'].idxmax(), ['Full_Name', 'value', 'kickoff_time',"team_code","news","selected"]]

mins=Xmins_data.loc[Xmins_data.groupby('name')['time'].idxmax(), ['name', 'average_minutes']]
print(mins)

pred_df=pd.read_csv("All_Predictions.csv").iloc[:,1:]

# Create a dictionary for quick lookup
name_value_dict = dict(zip(result['Full_Name'], result['value']))

name_team_dict= dict(zip(result['Full_Name'], result['team_code']))

name_news_dict= dict(zip(result['Full_Name'], result['news']))

name_selected_dict= dict(zip(result['Full_Name'], result['selected']))
print(name_selected_dict)
# Function to find the best fuzzy match
def get_best_match(name, choices, threshold=80):
    # ExtractOne returns (match, score, index), so we take only the first two
    match_data = process.extractOne(name, choices)
    if match_data:
        match, score, _ = match_data
        if score >= threshold:
            return match
    return None

# Get the best match for each name in pred_df
pred_df['Matched_Name'] = pred_df['Name'].apply(lambda x: get_best_match(x, result['Full_Name']))

# Map the value from result to pred_df based on the best match
pred_df['value'] = pred_df['Matched_Name'].map(name_value_dict)/10
pred_df['team'] = pred_df['Matched_Name'].map(name_team_dict)
pred_df['news'] = pred_df['Matched_Name'].map(name_news_dict)
pred_df['selected'] = pred_df['Matched_Name'].map(name_selected_dict)

def process_news(text):
    if pd.isna(text) or text.strip() == "":  # Blank check
        return 1
    elif "%" in text:
        # Extract number before %
        import re
        match = re.search(r"(\d+)%", text)
        if match:
            return int(match.group(1))/100  # Return the number before %
    return 0  # Default to 0 if no %

# Apply function to create new column
pred_df['offset'] = pred_df['news'].apply(process_news)
#pred_df[['p1', 'p2', 'p3','p4','p5','p6','p7','p8']] = pred_df[['p1', 'p2', 'p3','p4','p5','p6','p7','p8']].mul(pred_df['offset'], axis=0)
pred_df = pd.merge(pred_df, mins, left_on='Name', right_on='name', how='left')

pred_df["selected"] = pred_df["selected"]/11000000
pred_df["minutes_multiplier"] = np.minimum(1, pred_df['average_minutes'] / 70)

cols = ["p1", "p2", "p3", "p4", "p5", "p6", "p7", "p8"]
for col in cols:
    pred_df[col] = np.where(pred_df["offset"] < 1, pred_df[col] * pred_df["offset"], pred_df[col] * pred_df["minutes_multiplier"])
    
#pred_df[['p1', 'p2', 'p3','p4','p5','p6','p7','p8']] = pred_df[['p1', 'p2', 'p3','p4','p5','p6','p7','p8']].mul(pred_df['minutes_multiplier'], axis=0)
pred_df['p0']=0
pred_df.to_csv("Optimize_players.csv")  

<>:6: SyntaxWarning: invalid escape sequence '\F'
<>:6: SyntaxWarning: invalid escape sequence '\F'
C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_22648\343094543.py:6: SyntaxWarning: invalid escape sequence '\F'
  df=pd.read_csv("Raw_Data_24\Fantasy_season_2024_data.csv")



[notice] A new release of pip is available: 24.0 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


                     name  average_minutes
27530     Aaron_Cresswell            70.20
5309         Aaron_Hickey             0.00
24283     Aaron_Ramsdale0            87.02
24335     Aaron_Ramsdale1             0.22
35095        Aaron_Ramsey             0.07
...                   ...              ...
32409        Zeki_Amdouni            34.56
22811  Álex_Moreno Lopera             0.81
37434      Çaglar_Söyüncü            17.93
9296       Đorđe_Petrović             0.00
27641    Łukasz_Fabiański             3.10

[881 rows x 2 columns]
{'Aaron_Anselmino': 325, 'Aaron_Cresswell': 222886, 'Aaron_Hickey': 1288, 'Aaron_Ramsdale': 136046, 'Aaron_Wan-Bissaka': 671695, 'Abdoulaye_Doucouré': 51781, 'Abdukodir_Khusanov': 44666, 'Abdul_Fatawu': 19352, 'Adam_Armstrong': 68804, 'Adam_Lallana': 2820, 'Adam_Smith': 13634, 'Adam_Webster': 14211, 'Adam_Wharton': 16235, 'Adama_Traoré': 187269, 'Adrian_Mazilu': 1252, 'Albert_Grønbæk': 449, 'Alejandro_Garnacho': 531630, 'Alejo_Véliz': 570, 'Alex_Iwobi': 53

In [20]:
import pandas as pd

# Load data
df = pd.read_csv("ML_training2.csv").iloc[:, 1:]
max_time = df["time"].max()

# Filter the data for the relevant time range
filter_df = df[(df["time"] < max_time - 8) & (df["time"] >= max_time - 28)]   

# Pivot the data so each player's points are in separate columns
df_pivot = filter_df.pivot(index="time", columns="name", values="total_points").fillna(2)

# Compute variance and mean for each player
variance_per_player = df_pivot.var()
mean_per_player = df_pivot.mean()

# Compute variance / mean²
stability_metric = variance_per_player / (mean_per_player ** 2)

# Create a DataFrame with 'name' and 'stability_metric' columns
stability_df = pd.DataFrame({
    "name": stability_metric.index,  
    "variance_over_mean_squared": stability_metric.values
})

# Save the results to a CSV file
stability_df = stability_df.sort_values(by="variance_over_mean_squared", ascending=False)
stability_df.to_csv("stability_metric.csv", index=False)

# Print the DataFrame
print(stability_df)



                          name  variance_over_mean_squared
455        Junior_Firpo Adames                    4.338008
57            Anel_Ahmedhodžić                    4.116670
343              Jack_Robinson                    3.524027
390               Jayden_Bogle                    3.429276
280  Gabriel_Fernando de Jesus                    3.074027
..                         ...                         ...
143          Chiedozie_Ogbene0                    0.026243
362           Jakub_Stolarczyk                    0.026243
268            Filip_Jørgensen                    0.021482
843            Wayne_Hennessey                    0.013149
69              Antonín_Kinsky                    0.013149

[881 rows x 2 columns]


In [16]:

import requests
import requests
team_id=544468


def get_transfers(team_id):
    transfers_url = f"https://fantasy.premierleague.com/api/entry/{team_id}/transfers/"
    response_transfers = requests.get(transfers_url)

    if response_transfers.status_code != 200:
        print(f"Error fetching transfers (Status Code: {response_transfers.status_code})")
        return None

    transfers_data = response_transfers.json()
    return transfers_data

# Example Usage
team_transfers = get_transfers(team_id)

import pandas as pd
df = pd.DataFrame(team_transfers)

active=[]
for i in range(len(df["element_in"])):
    element_in=df["element_in"].values[-i-1]
    out_list=df["element_out"].iloc[0:-i-1].values
    if(element_in in out_list):
        active.append(0)
    else:
        active.append(1)
df["Active"]= list(reversed(active))

df=df[df["Active"]==1]
df=df[["element_in", "element_in_cost"]]

team_id = team_id  # Replace with your FPL team ID
gameweek = 28  # Replace with the desired gameweek

# API Endpoint
url = f"https://fantasy.premierleague.com/api/entry/{team_id}/event/{gameweek}/picks/"

# Request Data
response = requests.get(url)

# Check if request is successful
if response.status_code == 200:
    team_selection = response.json()
    picks=team_selection.get("picks")  # View the JSON response
    pick_df = pd.DataFrame(picks)
    
else:
    print(f"Error fetching team selection (Status Code: {response.status_code})")
print(df)

for g in range(len(pick_df)):
    element=pick_df["element"].values[g]
    if(element in [109]):
        element=304
    if(element not in df["element_in"].values):
        new_row = pd.DataFrame({'element_in': [element], 'element_in_cost': [np.nan]}, index=[len(df)])
        df = pd.concat([df, new_row], ignore_index=True)
print(df)

data=pd.read_csv("Raw_Data_24\Fantasy_season_2024_data.csv")
data=data[["Full_Name","element", "value", "kickoff_time"]]
data['kickoff_time'] = pd.to_datetime(data['kickoff_time'])
result = data.loc[data.groupby('Full_Name')['kickoff_time'].idxmax(), ['Full_Name','element', 'value', 'kickoff_time']]

team_df=pd.merge(df, result, left_on='element_in', right_on='element', how='left')
team_df['element_in_cost'] = team_df['element_in_cost'].fillna(team_df['value'])
team_df["selling_price_value"] = np.floor((team_df["value"] - team_df["element_in_cost"]) / 2).clip(lower=0)
team_df["selling_price"] = (team_df[["value", "element_in_cost"]].min(axis=1)+team_df["selling_price_value"])/10
print(team_df)
pred_data=pd.read_csv("All_Predictions.csv").iloc[:,1:]["Name"]
team_df=team_df[team_df["element_in_cost"]>30]
new_Names=[]
name_list=pred_data.values
for j in range(len(team_df)):
    name=team_df["Full_Name"].values[j]
    if(name in name_list):
        new_Names.append(name)
    elif(name+'1' in name_list):
        new_Names.append(name+'1')
    elif(name+'0' in name_list):
        new_Names.append(name+'0')
        
team_df["Full_Name"]=new_Names     

team_df.to_csv("Squad_data.csv")



print(team_df)
team_id = team_id  # Replace with your FPL team ID
url = f"https://fantasy.premierleague.com/api/entry/{team_id}/"
response = requests.get(url)
if response.status_code == 200:
    data = response.json()
    money_in_bank = data.get("last_deadline_bank", 0)/10  # Convert to actual value
else:
    print(f"Error fetching data (Status Code: {response.status_code})")



data = pd.read_csv("Optimize_players.csv")
players = data['Name'].tolist()
costs = data['value'].tolist()

squad=[]

for t in range (len(team_df)):
    name=team_df["Full_Name"].values[t]
    squad.append(players.index(name))
print(squad)

list1 = costs.copy()
list2 = team_df["selling_price"].values
# Update list1 with values from list2 at positions specified by indexes
for i in range(len(list2)):
    list1[squad[i]] = list2[i]  #

for j in range(len(list2)):
    print(list1[squad[j]])


<>:72: SyntaxWarning: invalid escape sequence '\F'
<>:72: SyntaxWarning: invalid escape sequence '\F'
C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_22648\2393417650.py:72: SyntaxWarning: invalid escape sequence '\F'
  data=pd.read_csv("Raw_Data_24\Fantasy_season_2024_data.csv")


    element_in  element_in_cost
0          447               72
1          533               47
2          741                8
3           74               51
5          235               50
6          580               44
8          231               43
10         211               47
11         366               84
12         110               62
13         311               70
14         401               85
16         328              131
20          78               56
22         182              109
    element_in  element_in_cost
0          447             72.0
1          533             47.0
2          741              8.0
3           74             51.0
4          235             50.0
5          580             44.0
6          231             43.0
7          211             47.0
8          366             84.0
9          110             62.0
10         311             70.0
11         401             85.0
12         328            131.0
13          78             56.0
14      

In [ ]:
WIldcard team

In [56]:
import pandas as pd
from pulp import LpMaximize, LpProblem, LpVariable, lpSum

# Load Data
data = pd.read_csv("Optimize_players.csv")

budget = 100.0  
sel_tresh=1*15
players = data['Name'].tolist()
positions = data['position'].tolist()
costs = data['value'].tolist()
teams = data['team'].tolist() 
selected = data['selected'].tolist() 
predicted_points = data[['p1', 'p2', 'p3','p4','p5','p6','p7','p8']].values

gameweeks = range(8)
num_players = len(players)

# Define Model
model = LpProblem("Maximize_Predicted_Points", LpMaximize)

# Decision Variables
x = {(i, t): LpVariable(cat='Binary', name=f"x_{i}_{t}") for i in range(num_players) for t in gameweeks}

# Bench Variables
bench = {(i, t): LpVariable(cat='Binary', name=f"bench_{i}_{t}") for i in range(num_players) for t in gameweeks}
bench_gk = {t: LpVariable(cat='Binary', name=f"bench_gk_{t}") for t in gameweeks}

# Transfer Tracking Variables
transfer_in = {(i, t): LpVariable(cat='Binary', name=f"transfer_in_{i}_{t}") for i in range(num_players) for t in range(1, 8)}
transfer_out = {(i, t): LpVariable(cat='Binary', name=f"transfer_out_{i}_{t}") for i in range(num_players) for t in range(1,8)}
saved_transfers = {t: LpVariable(cat='Integer', lowBound=0, upBound=3, name=f"saved_transfers_{t}") for t in range(8)}

# Decision Variable for Playing Status
y = {(i, t): LpVariable(cat='Binary', name=f"y_{i}_{t}") for i in range(num_players) for t in gameweeks}

for t in gameweeks:
    model += lpSum(y[i, t] for i in range(num_players) if positions[i] == 'DEF') == 3

# Objective: Maximize Total Points (only for playing players)
model += lpSum(y[i, t] * predicted_points[i][t] for i in range(num_players) for t in gameweeks)

# Ensure y is 1 only when player is in the squad and not benched
for t in gameweeks:
    for i in range(num_players):
        model += y[i, t] <= x[i, t]                  # Can only play if in squad
        model += y[i, t] <= 1 - bench[i, t]           # Can't play if benched
        model += y[i, t] >= x[i, t] + (1 - bench[i, t]) - 1  # Consistency

# Selected Constraint
for t in gameweeks:
    model += lpSum(x[i, t] * selected[i] for i in range(num_players)) <= sel_tresh

# Budget Constraint
for t in gameweeks:
    model += lpSum(x[i, t] * costs[i] for i in range(num_players)) <= budget
    
# Max 3 Players per Team Constraint
for t in gameweeks:
    for team in set(teams):
        model += lpSum(x[i, t] for i in range(num_players) if teams[i] == team) <= 3

# Total Players Constraint
for t in gameweeks:
    model += lpSum(x[i, t] for i in range(num_players)) == 15


# Position Constraints for each Gameweek
for t in gameweeks:
    model += lpSum(x[i, t] for i in range(num_players) if positions[i] == 'DEF') == 5  # 5 Defenders
    model += lpSum(x[i, t] for i in range(num_players) if positions[i] == 'GK') == 2   # 2 Goalkeepers
    model += lpSum(x[i, t] for i in range(num_players) if positions[i] == 'MID') == 5  # 5 Midfielders
    model += lpSum(x[i, t] for i in range(num_players) if positions[i] == 'FWD') == 3  # 3 Attackers

# Bench Constraints
for t in gameweeks:
    # Exactly 1 goalkeeper on the bench
    model += lpSum(bench[i, t] for i in range(num_players) if positions[i] == 'GK') == 1
    # Exactly 3 outfield players on the bench
    model += lpSum(bench[i, t] for i in range(num_players) if positions[i] != 'GK') == 3
    
    for i in range(num_players):
        # A player can only be benched if they are in the squad
        model += bench[i, t] <= x[i, t]

# Transfer Constraints
for t in range(1, 8):
    for i in range(num_players):
        model += transfer_in[i, t] >= x[i, t] - x[i, t - 1]
        model += transfer_out[i, t] >= x[i, t - 1] - x[i, t]
        model += transfer_out[i, t] <= x[i, t - 1]

    # Number of transfers allowed per week (considering saved transfers)
    model += lpSum(transfer_in[i, t] for i in range(num_players)) <= 1 + saved_transfers[t - 1]

    # Define saved transfers: If 1 or 0 transfers used, they are saved for the next week (capped at 2)
    model += saved_transfers[t] == saved_transfers[t - 1] + (1 - lpSum(transfer_in[i, t] for i in range(num_players)))
    model += saved_transfers[t] <= 3  

# Initial Transfers (Gameweek 1 starts with 1 available transfer)
model += saved_transfers[0] == 0  

# Solve the Model
model.solve()

# Check the status of the solution
print(f"Status: {model.status}")

# Display selected players for each gameweek
for t in gameweeks:
    print(f"\nGameweek {t+1} Squad:")
    for i in range(num_players):
        if x[i, t].varValue > 0.5:
            status = "Bench" if bench[i, t].varValue > 0.5 else "Playing"
            print(f"- {players[i]} ({positions[i]}) - {status}")

# Display Transfers for Each Gameweek
for t in range(1, 8):  
    print(f"\nTransfers for Gameweek {t+1}:")
    players_in = [players[i] for i in range(num_players) if transfer_in[i, t].varValue > 0.5]
    players_out = [players[i] for i in range(num_players) if transfer_out[i, t].varValue > 0.5]

    if players_in or players_out:
        print(f"  In: {', '.join(players_in) if players_in else 'None'}")
        print(f"  Out: {', '.join(players_out) if players_out else 'None'}")
    else:
        print("  No transfers this week.")


Status: 1

Gameweek 1 Squad:
- João_Pedro Junqueira de Jesus (FWD) - Playing
- Alexander_Isak (FWD) - Bench
- Chris_Wood0 (FWD) - Playing
- Daniel_Muñoz (DEF) - Bench
- Trent_Alexander-Arnold (DEF) - Bench
- Joško_Gvardiol (DEF) - Playing
- Neco_Williams (DEF) - Playing
- Rayan_Aït-Nouri (DEF) - Playing
- Justin_Kluivert (MID) - Playing
- Bryan_Mbeumo (MID) - Playing
- Sávio_'Savinho' Moreira de Oliveira (MID) - Playing
- Bruno_Borges Fernandes (MID) - Playing
- Morgan_Gibbs-White (MID) - Playing
- Kepa_Arrizabalaga0 (GK) - Playing
- Nick_Pope (GK) - Bench

Gameweek 2 Squad:
- João_Pedro Junqueira de Jesus (FWD) - Playing
- Alexander_Isak (FWD) - Playing
- Chris_Wood0 (FWD) - Playing
- Daniel_Muñoz (DEF) - Playing
- Trent_Alexander-Arnold (DEF) - Playing
- Joško_Gvardiol (DEF) - Playing
- Neco_Williams (DEF) - Bench
- Rayan_Aït-Nouri (DEF) - Bench
- Justin_Kluivert (MID) - Playing
- Bryan_Mbeumo (MID) - Playing
- Mohamed_Salah (MID) - Playing
- Sávio_'Savinho' Moreira de Oliveira (MID)

In [ ]:
FREE HIT

In [99]:
import pandas as pd
from pulp import LpMaximize, LpProblem, LpVariable, lpSum

# Load Data
data = pd.read_csv("Optimize_players.csv")

budget = 100.0  
sel_tresh = 1 * 15  # Max selected players
players = data['Name'].tolist()
positions = data['position'].tolist()
costs = data['value'].tolist()
teams = data['team'].tolist() 
selected = data['selected'].tolist() 
predicted_points = data[['p1']].values.flatten()  # Only use 'p1' for Gameweek 1

num_players = len(players)

# Define Model
model = LpProblem("Maximize_Predicted_Points_One_Round", LpMaximize)

# Decision Variables
x = {i: LpVariable(cat='Binary', name=f"x_{i}") for i in range(num_players)}  # Selected players
bench = {i: LpVariable(cat='Binary', name=f"bench_{i}") for i in range(num_players)}  # Bench players
y = {i: LpVariable(cat='Binary', name=f"y_{i}") for i in range(num_players)}  # Playing players

# Objective: Maximize Total Points for Gameweek 1
model += lpSum(y[i] * predicted_points[i] for i in range(num_players))

# Ensure y is 1 only when player is in the squad and not benched
for i in range(num_players):
    model += y[i] <= x[i]               # Can only play if selected in squad
    model += y[i] <= 1 - bench[i]        # Can't play if benched
    model += y[i] >= x[i] + (1 - bench[i]) - 1  # Consistency

# Selected Constraint
model += lpSum(x[i] * selected[i] for i in range(num_players)) <= sel_tresh


model += lpSum(y[i] for i in range(num_players) if positions[i] == 'DEF') == 3

# Budget Constraint
model += lpSum(x[i] * costs[i] for i in range(num_players)) <= budget

# Max 3 Players per Team Constraint
for team in set(teams):
    model += lpSum(x[i] for i in range(num_players) if teams[i] == team) <= 3

# Total Players Constraint (15 players in squad)
model += lpSum(x[i] for i in range(num_players)) == 15

# Position Constraints
model += lpSum(x[i] for i in range(num_players) if positions[i] == 'DEF') == 5  # 5 Defenders
model += lpSum(x[i] for i in range(num_players) if positions[i] == 'GK') == 2   # 2 Goalkeepers
model += lpSum(x[i] for i in range(num_players) if positions[i] == 'MID') == 5  # 5 Midfielders
model += lpSum(x[i] for i in range(num_players) if positions[i] == 'FWD') == 3  # 3 Attackers

# Bench Constraints
model += lpSum(bench[i] for i in range(num_players) if positions[i] == 'GK') == 1  # Exactly 1 GK on bench
model += lpSum(bench[i] for i in range(num_players) if positions[i] != 'GK') == 3  # Exactly 3 outfield players on bench

# A player can only be benched if they are in the squad
for i in range(num_players):
    model += bench[i] <= x[i]

# Solve the Model
model.solve()

# Check the status of the solution
print(f"Status: {model.status}")

# Display selected players
print("\nGameweek 1 Squad:")
for i in range(num_players):
    if x[i].varValue > 0.5:
        status = "Bench" if bench[i].varValue > 0.5 else "Playing"
        print(f"- {players[i]} ({positions[i]}) - {status}")


[6.5, 7.7, 5.7, 8.9, 5.4, 5.6, 5.9, 6.5, 5.5, 5.5, 5.5, 7.7, 5.9, 7.5, 5.3, 5.1, 5.4, 5.3, 5.5, 5.5, 5.0, 4.7, 5.6, 5.4, 5.1, 4.8, 5.3, 7.0, 7.5, 14.7, 6.9, 6.5, 9.4, 6.9, 5.5, 7.2, 4.9, 5.1, 4.9, 4.4, 7.3, 6.8, 5.3, 4.9, 6.7, 7.0, 5.2, 7.2, 6.3, 5.6, 4.8, 6.2, 4.3, 6.1, 4.8, 5.8, 4.4, 4.5, 4.5, 4.4, 4.4, 4.2, 4.3, 3.9, 5.1, 4.6, 4.4, 4.4, 4.5, 4.4, 4.3, 4.2, 4.4, 4.3, 4.1, 4.2, 4.9, 4.3, 4.4, 4.5, 4.4, 4.3, 4.4, 4.2, 5.2, 4.9, 4.8, 4.4, 4.4, 4.4, 4.3, 4.7, 4.8, 5.0, 4.5, 4.8, 4.0, 3.9, 4.4, 4.4, 4.9, 4.5, 4.2, 4.5, 5.0, 4.4, 3.8, 4.0, 4.3, 3.9, 3.9, 3.9, 3.9, 4.1, 4.4, 4.0, 4.1, 3.9, 3.9, 7.5, 4.7, 4.7, 5.2, 3.9, 5.8, 4.6, 6.4, 5.3, 5.3, 5.9, 4.3, 5.4, 5.3, 5.1, 5.0, 4.3, 4.9, 4.4, 4.3, 4.9, 4.3, 4.8, 4.3, 4.4, 4.4, 4.3, 4.5, 5.4, 5.6, 5.3, 4.7, 4.4, 4.2, 4.8, 4.5, 3.9, 4.0, 4.3, 4.0, 4.2, 4.3, 4.0, 4.3, 5.3, 4.9, 4.4, 4.8, 4.5, 4.1, 3.9, 4.4, 4.3, 4.2, 4.7, 3.8, 4.5, 4.3, 4.3, 4.7, 6.5, 4.6, 8.2, 6.2, 10.2, 4.9, 6.8, 6.7, 4.5, 6.0, 6.2, 5.2, 5.1, 5.0, 5.2, 5.4, 5.6, 5.5, 4.8, 5.3, 5.

In [55]:
import pandas as pd
from pulp import LpMaximize, LpProblem, LpVariable, lpSum, PULP_CBC_CMD

# Load Data
data = pd.read_csv("Optimize_players.csv")

budget = 103.0  
players = data['Name'].tolist()
positions = data['position'].tolist()
costs = data['value'].tolist()
teams = data['team'].tolist() 
predicted_points = data[['p0','p1', 'p2', 'p3','p4','p5','p6','p7','p8']].values


initial_squad=[players.index('Cody_Gakpo'),players.index('Alexander_Isak'),players.index('Yoane_Wissa'),players.index('Antoine_Semenyo'),
               players.index('Mohamed_Salah'),players.index('Dango_Ouattara'),players.index('Bruno_Borges Fernandes'),players.index('Cole_Palmer0'),
               players.index('Trent_Alexander-Arnold'),players.index('Daniel_Muñoz'),players.index('Dean_Huijsen'),players.index('Vitalii_Mykolenko'),
               players.index('Jacob_Greaves'),players.index('Jordan_Pickford'),players.index('Łukasz_Fabiański')]

#optimize_range=5
optimize_range=9
gameweeks = range(optimize_range)
num_players = len(players)

# Define Model
model = LpProblem("Maximize_Predicted_Points", LpMaximize)

# Decision Variables
x = {(i, t): LpVariable(cat='Binary', name=f"x_{i}_{t}") for i in range(num_players) for t in gameweeks}

for i in range(num_players):
    if i in initial_squad:
        model += x[i, 0] == 1
    else:
        model += x[i, 0] == 0

# Bench Variables
bench = {(i, t): LpVariable(cat='Binary', name=f"bench_{i}_{t}") for i in range(num_players) for t in gameweeks}
bench_gk = {t: LpVariable(cat='Binary', name=f"bench_gk_{t}") for t in gameweeks}

# Transfer Tracking Variables
transfer_in = {(i, t): LpVariable(cat='Binary', name=f"transfer_in_{i}_{t}") for i in range(num_players) for t in range(0, optimize_range)}
transfer_out = {(i, t): LpVariable(cat='Binary', name=f"transfer_out_{i}_{t}") for i in range(num_players) for t in range(0, optimize_range)}
saved_transfers = {t: LpVariable(cat='Integer', lowBound=0, upBound=3, name=f"saved_transfers_{t}") for t in range(optimize_range)}

c = {(i, t): LpVariable(cat='Binary', name=f"captain_{i}_{t}") for i in range(num_players) for t in gameweeks}

        
# Decision Variable for Playing Status
y = {(i, t): LpVariable(cat='Binary', name=f"y_{i}_{t}") for i in range(num_players) for t in gameweeks}

for t in gameweeks:
    model += lpSum(y[i, t] for i in range(num_players) if positions[i] == 'DEF') == 3

# Objective: Maximize Total Points (only for playing players)
model += lpSum((y[i, t] + c[i, t]) * predicted_points[i][t] for i in range(num_players) for t in range(1, optimize_range))

for t in range(0, optimize_range):
    model += lpSum(transfer_in[i, t] for i in range(num_players)) == lpSum(transfer_out[i, t] for i in range(num_players))
for t in range(1, optimize_range):
    for i in range(num_players):
        model += x[i, t] >= x[i, t - 1] - transfer_out[i, t]  # If not transferred out, stays in squad
        
# Ensure y is 1 only when player is in the squad and not benched
for t in gameweeks:
    for i in range(num_players):
        model += y[i, t] <= x[i, t]                  # Can only play if in squad
        model += y[i, t] <= 1 - bench[i, t]           # Can't play if benched
        model += y[i, t] >= x[i, t] + (1 - bench[i, t]) - 1  # Consistency

#captancy
for t in gameweeks:
    model += lpSum(c[i, t] for i in range(num_players)) == 1  # Only one captain per GW

for t in gameweeks:
    for i in range(num_players):
        model += c[i, t] <= y[i, t]  # Captain must be a playing player
        
    
# Budget Constraint
for t in gameweeks:
    model += lpSum(x[i, t] * costs[i] for i in range(num_players)) <= budget

# Max 3 Players per Team Constraint
for t in gameweeks:
    for team in set(teams):
        model += lpSum(x[i, t] for i in range(num_players) if teams[i] == team) <= 3

# Total Players Constraint
for t in gameweeks:
    model += lpSum(x[i, t] for i in range(num_players)) == 15


# Position Constraints for each Gameweek
for t in gameweeks:
    model += lpSum(x[i, t] for i in range(num_players) if positions[i] == 'DEF') == 5  # 5 Defenders
    model += lpSum(x[i, t] for i in range(num_players) if positions[i] == 'GK') == 2   # 2 Goalkeepers
    model += lpSum(x[i, t] for i in range(num_players) if positions[i] == 'MID') == 5  # 5 Midfielders
    model += lpSum(x[i, t] for i in range(num_players) if positions[i] == 'FWD') == 3  # 3 Attackers

# Bench Constraints
for t in gameweeks:
    # Exactly 1 goalkeeper on the bench
    model += lpSum(bench[i, t] for i in range(num_players) if positions[i] == 'GK') == 1
    # Exactly 3 outfield players on the bench
    model += lpSum(bench[i, t] for i in range(num_players) if positions[i] != 'GK') == 3
    
    for i in range(num_players):
        # A player can only be benched if they are in the squad
        model += bench[i, t] <= x[i, t]

# Transfer Constraints
for t in gameweeks[1:]:
    for i in range(num_players):
        model += transfer_in[i, t] >= x[i, t] - x[i, t - 1]
        model += transfer_out[i, t] >= x[i, t - 1] - x[i, t]
        model += transfer_out[i, t] <= x[i, t - 1]

    # Number of transfers allowed per week (considering saved transfers)
    model += lpSum(transfer_in[i, t] for i in range(num_players)) <= 1 + saved_transfers[t - 1]

    # Define saved transfers: If 1 or 0 transfers used, they are saved for the next week (capped at 2)
    model += saved_transfers[t] == saved_transfers[t - 1] + (1 - lpSum(transfer_in[i, t] for i in range(num_players)))
    model += saved_transfers[t] <= 3  

# Initial Transfers (Gameweek 1 starts with 1 available transfer)
model += saved_transfers[0] == 0  

# Solve the Model
model.solve()

# Check the status of the solution
print(f"Status: {model.status}")

# Display selected players for each gameweek
for t in range(1, optimize_range):
    print(f"\nGameweek {t+27} Squad:")
    for i in range(num_players):
        if x[i, t].varValue > 0.5:
            status = "Bench" if bench[i, t].varValue > 0.5 else "Playing"
            print(f"- {players[i]} ({positions[i]}) - {status}")

# Display Transfers for Each Gameweek
for t in range(1, optimize_range):  
    print(f"\nTransfers for Gameweek {t+27}:")
    players_in = [players[i] for i in range(num_players) if transfer_in[i, t].varValue > 0.5]
    players_out = [players[i] for i in range(num_players) if transfer_out[i, t].varValue > 0.5]

    if players_in or players_out:
        print(f"  In: {', '.join(players_in) if players_in else 'None'}")
        print(f"  Out: {', '.join(players_out) if players_out else 'None'}")
    else:
        print("  No transfers this week.")


for t in range(1, optimize_range):
    for i in range(num_players):
        if c[i, t].varValue > 0.5:
            print(f"Gameweek {t} Captain: {players[i]}")

Status: 1

Gameweek 28 Squad:
- Yoane_Wissa (FWD) - Playing
- Cody_Gakpo (FWD) - Bench
- Alexander_Isak (FWD) - Playing
- Dean_Huijsen (DEF) - Playing
- Daniel_Muñoz (DEF) - Playing
- Vitalii_Mykolenko (DEF) - Bench
- Jacob_Greaves (DEF) - Bench
- Trent_Alexander-Arnold (DEF) - Playing
- Justin_Kluivert (MID) - Playing
- Dango_Ouattara (MID) - Playing
- Cole_Palmer0 (MID) - Playing
- Mohamed_Salah (MID) - Playing
- Bruno_Borges Fernandes (MID) - Playing
- Jordan_Pickford (GK) - Playing
- Łukasz_Fabiański (GK) - Bench

Gameweek 29 Squad:
- Yoane_Wissa (FWD) - Playing
- Alexander_Isak (FWD) - Playing
- Matheus_Santos Carneiro Da Cunha (FWD) - Playing
- Dean_Huijsen (DEF) - Playing
- Daniel_Muñoz (DEF) - Bench
- Vitalii_Mykolenko (DEF) - Playing
- Jacob_Greaves (DEF) - Playing
- Trent_Alexander-Arnold (DEF) - Bench
- Justin_Kluivert (MID) - Playing
- Dango_Ouattara (MID) - Playing
- Cole_Palmer0 (MID) - Playing
- Mohamed_Salah (MID) - Bench
- Bruno_Borges Fernandes (MID) - Playing
- Jorda

In [ ]:
Team_Optimize

In [25]:
import requests

def get_transfers(team_id):
    transfers_url = f"https://fantasy.premierleague.com/api/entry/{182285}/transfers/"
    response_transfers = requests.get(transfers_url)

    if response_transfers.status_code != 200:
        print(f"Error fetching transfers (Status Code: {response_transfers.status_code})")
        return None

    transfers_data = response_transfers.json()
    return transfers_data

# Example Usage
team_transfers = get_transfers(team_id)

import pandas as pd
df = pd.DataFrame(team_transfers)
print(df)


transfers1 = df.groupby('event').size().reset_index(name='count')
max_event = 28
print("Max event:", max_event)
saved_transfers = 0
last_event = 0

for h in range(max_event):
    new_event = last_event + 1
    if new_event in transfers1["event"].values:
        ind = transfers1["event"].tolist().index(new_event)
        transfers_made = transfers1["count"].values[ind]
        saved_transfers = saved_transfers - transfers_made
        saved_transfers = max(0, saved_transfers)
    else:
        saved_transfers += 1
    last_event = new_event
saved_transfers+=1
print("Saved transfers:", saved_transfers)





    element_in  element_in_cost  element_out  element_out_cost   entry  event  \
0          741                8          737                 8  182285     26   
1           74               51          327                75  182285     26   
2           12               46           54                53  182285     26   
3          737                8          742                 5  182285     25   
4           99               78          398                75  182285     24   
5          211               48          355                45  182285     23   
6          447               69          180                80  182285     22   
7          398               73          366                84  182285     19   
8          473               44          447                63  182285     18   
9          311               71          533                48  182285     18   
10         327               75           99                74  182285     18   
11         180              

In [ ]:
import pandas as pd
from pulp import LpMaximize, LpProblem, LpVariable, lpSum, PULP_CBC_CMD
import requests
import numpy as np

#eliot-239743, Aria-182285
team_id=544468
wildcard_round = 3  # Gameweek 3 (Index t=2)
bench_points_gw=20
Last_GW=28
initial_saved=1


def get_transfers(team_id):
    transfers_url = f"https://fantasy.premierleague.com/api/entry/{team_id}/transfers/"
    response_transfers = requests.get(transfers_url)

    if response_transfers.status_code != 200:
        print(f"Error fetching transfers (Status Code: {response_transfers.status_code})")
        return None

    transfers_data = response_transfers.json()
    return transfers_data

# Example Usage
team_transfers = get_transfers(team_id)

import pandas as pd
df = pd.DataFrame(team_transfers)

active=[]
for i in range(len(df["element_in"])):
    element_in=df["element_in"].values[-i-1]
    out_list=df["element_out"].iloc[0:-i-1].values
    if(element_in in out_list):
        active.append(0)
    else:
        active.append(1)
df["Active"]= list(reversed(active))

df=df[df["Active"]==1]
df=df[["element_in", "element_in_cost"]]

team_id = team_id  # Replace with your FPL team ID
gameweek = Last_GW  # Replace with the desired gameweek

# API Endpoint
url = f"https://fantasy.premierleague.com/api/entry/{team_id}/event/{gameweek}/picks/"

# Request Data
response = requests.get(url)

# Check if request is successful
if response.status_code == 200:
    team_selection = response.json()
    picks=team_selection.get("picks")  # View the JSON response
    pick_df = pd.DataFrame(picks)
    
else:
    print(f"Error fetching team selection (Status Code: {response.status_code})")
print(df)

for g in range(len(pick_df)):
    element=pick_df["element"].values[g]
    if(element in [109]):
        element=304
    if(element not in df["element_in"].values):
        new_row = pd.DataFrame({'element_in': [element], 'element_in_cost': [np.nan]}, index=[len(df)])
        df = pd.concat([df, new_row], ignore_index=True)
print(df)

data=pd.read_csv("Raw_Data_24\Fantasy_season_2024_data.csv")
data=data[["Full_Name","element", "value", "kickoff_time"]]
data['kickoff_time'] = pd.to_datetime(data['kickoff_time'])
result = data.loc[data.groupby('Full_Name')['kickoff_time'].idxmax(), ['Full_Name','element', 'value', 'kickoff_time']]

team_df=pd.merge(df, result, left_on='element_in', right_on='element', how='left')
team_df['element_in_cost'] = team_df['element_in_cost'].fillna(team_df['value'])
team_df["selling_price_value"] = np.floor((team_df["value"] - team_df["element_in_cost"]) / 2).clip(lower=0)
team_df["selling_price"] = (team_df[["value", "element_in_cost"]].min(axis=1)+team_df["selling_price_value"])/10
print(team_df)
pred_data=pd.read_csv("All_Predictions.csv").iloc[:,1:]["Name"]
team_df=team_df[team_df["element_in_cost"]>30]
new_Names=[]
name_list=pred_data.values
for j in range(len(team_df)):
    name=team_df["Full_Name"].values[j]
    if(name in name_list):
        new_Names.append(name)
    elif(name+'1' in name_list):
        new_Names.append(name+'1')
    elif(name+'0' in name_list):
        new_Names.append(name+'0')
        
team_df["Full_Name"]=new_Names     

team_df.to_csv("Squad_data.csv")


# Load Data
data = pd.read_csv("Optimize_players.csv")
squad=pd.read_csv("Squad_data.csv")

url = f"https://fantasy.premierleague.com/api/entry/{team_id}/"
response = requests.get(url)
if response.status_code == 200:
    resonsep_data = response.json()
    
else:
    print(f"Error fetching data (Status Code: {response.status_code})")

money_in_bank_init = resonsep_data.get("last_deadline_bank", 0)/10  # Convert to actual value


players = data['Name'].tolist()
costs = data['value'].tolist()
initial_squad=[]
for t in range (len(squad)):
    name=squad["Full_Name"].values[t]
    initial_squad.append(players.index(name))


list1 = costs.copy()
selling_cost = squad["selling_price"].values

budget_amount=sum(selling_cost)+money_in_bank_init
print(budget_amount)
# Update list1 with values from list2 at positions specified by indexes
for i in range(len(selling_cost)):
    list1[initial_squad[i]] = selling_cost[i]  

# Define Constants
#budget = 103.0  
players = data['Name'].tolist()
positions = data['position'].tolist()
costs = data['value'].tolist()
teams = data['team'].tolist()
predicted_points = data[['p0', 'p1', 'p2', 'p3', 'p4', 'p5', 'p6', 'p7', 'p8']].values
# Initial Squad
"""initial_squad=[players.index('Chris_Wood0'),players.index('Alexander_Isak'),players.index('Yoane_Wissa'),players.index('Antoine_Semenyo'),
               players.index('Mohamed_Salah'),players.index('Dango_Ouattara'),players.index('Bruno_Borges Fernandes'),players.index('Cole_Palmer0'),
               players.index('Trent_Alexander-Arnold'),players.index('Daniel_Muñoz'),players.index('Dean_Huijsen'),players.index('Vitalii_Mykolenko'),
               players.index('Rayan_Aït-Nouri'),players.index('Jordan_Pickford'),players.index('Łukasz_Fabiański')]"""


# Define Gameweeks
optimize_range = 9  # Number of gameweeks to optimize
gameweeks = range(optimize_range)
num_players = len(players)

# Define Wildcard Round (where unlimited transfers are allowed)

# Define Model
model = LpProblem("Maximize_Predicted_Points", LpMaximize)

# Decision Variables
x = {(i, t): LpVariable(cat='Binary', name=f"x_{i}_{t}") for i in range(num_players) for t in gameweeks}
bench = {(i, t): LpVariable(cat='Binary', name=f"bench_{i}_{t}") for i in range(num_players) for t in gameweeks}
c = {(i, t): LpVariable(cat='Binary', name=f"captain_{i}_{t}") for i in range(num_players) for t in gameweeks}
y = {(i, t): LpVariable(cat='Binary', name=f"y_{i}_{t}") for i in range(num_players) for t in gameweeks}
transfer_in = {(i, t): LpVariable(cat='Binary', name=f"transfer_in_{i}_{t}") for i in range(num_players) for t in gameweeks}
transfer_out = {(i, t): LpVariable(cat='Binary', name=f"transfer_out_{i}_{t}") for i in range(num_players) for t in gameweeks}
saved_transfers = {t: LpVariable(cat='Integer', lowBound=0, upBound=3, name=f"saved_transfers_{t}") for t in gameweeks}
money_in_bank_var = {t: LpVariable(f"money_in_bank_{t}", lowBound=0, cat='Continuous') for t in gameweeks}

# Initial Squad Constraint (Gameweek 1)
for i in range(num_players):
    model += x[i, 0] == (1 if i in initial_squad else 0)

# Objective: Maximize Total Points (only for playing players)
#model += lpSum((y[i, t] + c[i, t]) * predicted_points[i][t] for i in range(num_players) for t in gameweeks)

model += lpSum(
    (y[i, t] + c[i, t]) * predicted_points[i][t] + 
    (bench[i, t] * predicted_points[i][t] if t == bench_points_gw else 0)
    for i in range(num_players) for t in gameweeks
)
# Position Constraints
for t in gameweeks:
    model += lpSum(x[i, t] for i in range(num_players)) == 15  # Squad size 15
    model += lpSum(x[i, t] for i in range(num_players) if positions[i] == 'DEF') == 5
    model += lpSum(x[i, t] for i in range(num_players) if positions[i] == 'GK') == 2
    model += lpSum(x[i, t] for i in range(num_players) if positions[i] == 'MID') == 5
    model += lpSum(x[i, t] for i in range(num_players) if positions[i] == 'FWD') == 3

# Bench Constraints (1 GK, 3 Outfield)
for t in gameweeks:
    model += lpSum(bench[i, t] for i in range(num_players) if positions[i] == 'GK') == 1
    model += lpSum(bench[i, t] for i in range(num_players) if positions[i] != 'GK') == 3
    for i in range(num_players):
        model += bench[i, t] <= x[i, t]


for t in gameweeks:
    model += lpSum(y[i, t] for i in range(num_players) if positions[i] == 'DEF') >= 3  # Ensure at least 3 defenders play

# Ensure playing status
for t in gameweeks:
    for i in range(num_players):
        model += y[i, t] <= x[i, t]
        model += y[i, t] <= 1 - bench[i, t]
        model += y[i, t] >= x[i, t] + (1 - bench[i, t]) - 1

# Captain Selection
for t in gameweeks:
    model += lpSum(c[i, t] for i in range(num_players)) == 1
    for i in range(num_players):
        model += c[i, t] <= y[i, t]

#Budget

for t in gameweeks[1:]:
        model += money_in_bank_var[t] == money_in_bank_var[t-1] + lpSum(transfer_out[i, t] * list1[i] for i in range(num_players)) - lpSum(transfer_in[i, t] * costs[i] for i in range(num_players))

    #model += money_in_bank_var[t] == money_in_bank_var[t - 1]+(lpSum(transfer_out[i, t]*list1[i] for i in range(num_players))) -(lpSum(transfer_in[i, t]*costs[i] for i in range(num_players)))
for t in gameweeks:
    model += lpSum(x[i, t] * list1[i] for i in range(num_players)) + money_in_bank_var[t] == budget_amount


# Max 3 Players per Team
for t in gameweeks:
    for team in set(teams):
        model += lpSum(x[i, t] for i in range(num_players) if teams[i] == team) <= 3

# Transfer Constraints
for t in gameweeks[1:]:
    if t == wildcard_round:
        # Wildcard round: No transfer limits
        for i in range(num_players):
            model += x[i, t] >= x[i, t - 1] - transfer_out[i, t]
            model += x[i, t] <= x[i, t - 1] + transfer_in[i, t]
    else:
        # Normal transfer constraints
        for i in range(num_players):
            model += transfer_in[i, t] >= x[i, t] - x[i, t - 1]
            model += transfer_out[i, t] >= x[i, t - 1] - x[i, t]
            model += transfer_out[i, t] <= x[i, t - 1]

        model += lpSum(transfer_in[i, t] for i in range(num_players)) <= 1 + saved_transfers[t - 1]
for t in gameweeks[1:]:
    if t == wildcard_round:
        model += saved_transfers[t] == 0  # Reset saved transfers after wildcard
    else:
        model += saved_transfers[t] == saved_transfers[t - 1] + (1 - lpSum(transfer_in[i, t] for i in range(num_players)))
        model += saved_transfers[t] <= 3  # Maximum of 3 saved transfers
        

# Initial Transfers
model += saved_transfers[0] == initial_saved 
model += money_in_bank_var[0] == money_in_bank_init

# Solve Model
model.solve()
#model.solve(PULP_CBC_CMD(msg=True))
# Display Results
print(f"Status: {model.status}")
print(f"\nTotal Optimized Predicted Points: {model.objective.value()}")
# Display Squad & Transfers
for t in range(1, optimize_range):
    print(f"\nGameweek {t+28} Squad:")
    for i in range(num_players):
        if x[i, t].varValue > 0.5:
            status = "Bench" if bench[i, t].varValue > 0.5 else "Playing"
            print(f"- {players[i]} ({positions[i]}) - {status}")

# Display Transfers for Each Gameweek
for t in range(1, optimize_range):  
    print(f"\nTransfers for Gameweek {t+28}:")
    if t == wildcard_round:
        print("  Wildcard Activated! All players can be changed.")
    else:
        players_in = [players[i] for i in range(num_players) if transfer_in[i, t].varValue > 0.5]
        players_out = [players[i] for i in range(num_players) if x[i, t-1].varValue > 0.5 and x[i, t].varValue < 0.5]

        if players_in or players_out:
            print(f"  In: {', '.join(players_in) if players_in else 'None'}")
            print(f"  Out: {', '.join(players_out) if players_out else 'None'}")
        else:
            print("  No transfers this week.")
# Display Captains
for t in range(1, optimize_range):
    for i in range(num_players):
        if c[i, t].varValue > 0.5:
            print(f"Gameweek {t+28} Captain: {players[i]}")

#569.8

<>:72: SyntaxWarning: invalid escape sequence '\F'
<>:72: SyntaxWarning: invalid escape sequence '\F'
C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_20060\2570932982.py:72: SyntaxWarning: invalid escape sequence '\F'
  data=pd.read_csv("Raw_Data_24\Fantasy_season_2024_data.csv")


    element_in  element_in_cost
0          447               72
1          533               47
2          741                8
3           74               51
5          235               50
6          580               44
8          231               43
10         211               47
11         366               84
12         110               62
13         311               70
14         401               85
16         328              131
20          78               56
22         182              109
    element_in  element_in_cost
0          447             72.0
1          533             47.0
2          741              8.0
3           74             51.0
4          235             50.0
5          580             44.0
6          231             43.0
7          211             47.0
8          366             84.0
9          110             62.0
10         311             70.0
11         401             85.0
12         328            131.0
13          78             56.0
14      

In [24]:
import pandas as pd
from pulp import LpMaximize, LpProblem, LpVariable, lpSum, PULP_CBC_CMD
from pulp import HiGHS_CMD
import requests
import numpy as np

#eliot-239743, Aria-182285, meg-544468
team_id=544468
wildcard_round = 3  # Gameweek 3 (Index t=2)
bench_points_gw=20
Last_GW=28
hit=1


def get_transfers(team_id):
    transfers_url = f"https://fantasy.premierleague.com/api/entry/{team_id}/transfers/"
    response_transfers = requests.get(transfers_url)

    if response_transfers.status_code != 200:
        print(f"Error fetching transfers (Status Code: {response_transfers.status_code})")
        return None

    transfers_data = response_transfers.json()
    return transfers_data

# Example Usage
team_transfers = get_transfers(team_id)

import pandas as pd
df = pd.DataFrame(team_transfers)

transfers1 = df.groupby('event').size().reset_index(name='count')
max_event = Last_GW
print("Max event:", max_event)
saved_transfers = 0
last_event = 0

for h in range(max_event):
    new_event = last_event + 1
    if new_event in transfers1["event"].values:
        ind = transfers1["event"].tolist().index(new_event)
        transfers_made = transfers1["count"].values[ind]
        saved_transfers = saved_transfers - transfers_made
        saved_transfers = max(0, saved_transfers)
    else:
        saved_transfers += 1
    last_event = new_event
initial_saved=saved_transfers+hit
print(initial_saved)

active=[]
for i in range(len(df["element_in"])):
    element_in=df["element_in"].values[-i-1]
    out_list=df["element_out"].iloc[0:-i-1].values
    if(element_in in out_list):
        active.append(0)
    else:
        active.append(1)
df["Active"]= list(reversed(active))

df=df[df["Active"]==1]
df=df[["element_in", "element_in_cost"]]

team_id = team_id  # Replace with your FPL team ID
gameweek = Last_GW  # Replace with the desired gameweek

# API Endpoint
url = f"https://fantasy.premierleague.com/api/entry/{team_id}/event/{gameweek}/picks/"

# Request Data
response = requests.get(url)

# Check if request is successful
if response.status_code == 200:
    team_selection = response.json()
    picks=team_selection.get("picks")  # View the JSON response
    pick_df = pd.DataFrame(picks)
    
else:
    print(f"Error fetching team selection (Status Code: {response.status_code})")

for g in range(len(pick_df)):
    element=pick_df["element"].values[g]
    if(element in [109,536]):
        df=df[df["element_in"]!=element]
        element=304
        
    elif(element in [162]):
        df=df[df["element_in"]!=element]
        element=163
        
    if(element not in df["element_in"].values):
        new_row = pd.DataFrame({'element_in': [element], 'element_in_cost': [np.nan]}, index=[len(df)])
        df = pd.concat([df, new_row], ignore_index=True)

data=pd.read_csv("Raw_Data_24\Fantasy_season_2024_data.csv")
data=data[["Full_Name","element", "value", "kickoff_time"]]
data['kickoff_time'] = pd.to_datetime(data['kickoff_time'])
result = data.loc[data.groupby('Full_Name')['kickoff_time'].idxmax(), ['Full_Name','element', 'value', 'kickoff_time']]

team_df=pd.merge(df, result, left_on='element_in', right_on='element', how='left')

team_df['element_in_cost'] = team_df['element_in_cost'].fillna(team_df['value'])
team_df["selling_price_value"] = np.floor((team_df["value"] - team_df["element_in_cost"]) / 2).clip(lower=0)
team_df["selling_price"] = (team_df[["value", "element_in_cost"]].min(axis=1)+team_df["selling_price_value"])/10
pred_data=pd.read_csv("All_Predictions.csv").iloc[:,1:]["Name"]
team_df=team_df[team_df["element_in_cost"]>30]
new_Names=[]
print(team_df)
name_list=pred_data.values
for j in range(len(team_df)):
    name=team_df["Full_Name"].values[j]
    if(name in name_list):
        new_Names.append(name)
    elif(name+'1' in name_list):
        new_Names.append(name+'1')
    elif(name+'0' in name_list):
        new_Names.append(name+'0')
print(new_Names)
team_df["Full_Name"]=new_Names     

team_df.to_csv("Squad_data.csv")


# Load Data
data = pd.read_csv("Optimize_players.csv")
squad=pd.read_csv("Squad_data.csv")

url = f"https://fantasy.premierleague.com/api/entry/{team_id}/"
response = requests.get(url)
if response.status_code == 200:
    resonsep_data = response.json()
    
else:
    print(f"Error fetching data (Status Code: {response.status_code})")

money_in_bank_init = resonsep_data.get("last_deadline_bank", 0)/10  # Convert to actual value


players = data['Name'].tolist()
costs = data['value'].tolist()
initial_squad=[]
for t in range (len(squad)):
    name=squad["Full_Name"].values[t]
    initial_squad.append(players.index(name))


list1 = costs.copy()
selling_cost = squad["selling_price"].values

budget_amount=sum(selling_cost)+money_in_bank_init
print(budget_amount)
# Update list1 with values from list2 at positions specified by indexes
for i in range(len(selling_cost)):
    list1[initial_squad[i]] = selling_cost[i]  

# Define Constants
#budget = 103.0  
players = data['Name'].tolist()
positions = data['position'].tolist()
costs = data['value'].tolist()
teams = data['team'].tolist()
predicted_points = data[['p0', 'p1', 'p2', 'p3', 'p4', 'p5', 'p6', 'p7', 'p8']].values
# Initial Squad
"""initial_squad=[players.index('Chris_Wood0'),players.index('Alexander_Isak'),players.index('Yoane_Wissa'),players.index('Antoine_Semenyo'),
               players.index('Mohamed_Salah'),players.index('Dango_Ouattara'),players.index('Bruno_Borges Fernandes'),players.index('Cole_Palmer0'),
               players.index('Trent_Alexander-Arnold'),players.index('Daniel_Muñoz'),players.index('Dean_Huijsen'),players.index('Vitalii_Mykolenko'),
               players.index('Rayan_Aït-Nouri'),players.index('Jordan_Pickford'),players.index('Łukasz_Fabiański')]"""


# --- Define Gameweeks and Precompute Indices ---
optimize_range = 9  # Number of gameweeks to optimize
gameweeks = range(optimize_range)
num_players = len(players)

# Precompute position indices so we don’t iterate over all players each time.
def_indices   = [i for i, pos in enumerate(positions) if pos == 'DEF']
gk_indices    = [i for i, pos in enumerate(positions) if pos == 'GK']
mid_indices   = [i for i, pos in enumerate(positions) if pos == 'MID']
fwd_indices   = [i for i, pos in enumerate(positions) if pos == 'FWD']
outfield_indices = [i for i, pos in enumerate(positions) if pos != 'GK']

# Precompute team indices: dictionary mapping team to list of player indices
teams_set = set(teams)
team_to_indices = {team: [i for i, t in enumerate(teams) if t == team] for team in teams_set}

# --- Define Model and Decision Variables ---
model = LpProblem("Maximize_Predicted_Points", LpMaximize)

# Decision Variables
x            = {(i, t): LpVariable(f"x_{i}_{t}", cat='Binary') for i in range(num_players) for t in gameweeks}
bench        = {(i, t): LpVariable(f"bench_{i}_{t}", cat='Binary') for i in range(num_players) for t in gameweeks}
c            = {(i, t): LpVariable(f"captain_{i}_{t}", cat='Binary') for i in range(num_players) for t in gameweeks}
y            = {(i, t): LpVariable(f"y_{i}_{t}", cat='Binary') for i in range(num_players) for t in gameweeks}
transfer_in  = {(i, t): LpVariable(f"transfer_in_{i}_{t}", cat='Binary') for i in range(num_players) for t in gameweeks}
transfer_out = {(i, t): LpVariable(f"transfer_out_{i}_{t}", cat='Binary') for i in range(num_players) for t in gameweeks}
saved_transfers   = {t: LpVariable(f"saved_transfers_{t}", lowBound=0, upBound=5, cat='Integer') for t in gameweeks}
money_in_bank_var = {t: LpVariable(f"money_in_bank_{t}", lowBound=0, cat='Continuous') for t in gameweeks}

# --- Initial Squad Constraint (Gameweek 0) ---
for i in range(num_players):
    model += x[i, 0] == (1 if i in initial_squad else 0)

# --- Objective Function ---
# (Bench points term is added only if bench_points_gw is in the gameweek range)
obj = lpSum((y[i, t] + c[i, t]) * predicted_points[i][t] 
            for i in range(num_players) for t in gameweeks)
if bench_points_gw in gameweeks:
    obj += lpSum(bench[i, bench_points_gw] * predicted_points[i][bench_points_gw] 
                 for i in range(num_players))
model += obj

# --- Position Constraints ---
for t in gameweeks:
    model += lpSum(x[i, t] for i in range(num_players)) == 15
    model += lpSum(x[i, t] for i in def_indices) == 5
    model += lpSum(x[i, t] for i in gk_indices) == 2
    model += lpSum(x[i, t] for i in mid_indices) == 5
    model += lpSum(x[i, t] for i in fwd_indices) == 3

# --- Bench Constraints ---
for t in gameweeks:
    model += lpSum(bench[i, t] for i in gk_indices) == 1
    model += lpSum(bench[i, t] for i in outfield_indices) == 3
    for i in range(num_players):
        model += bench[i, t] <= x[i, t]

# --- Playing Status Constraints ---
for t in gameweeks:
    # Ensure at least 3 defenders are in the starting XI
    model += lpSum(y[i, t] for i in def_indices) >= 3
    for i in range(num_players):
        model += y[i, t] <= x[i, t]
        model += y[i, t] <= 1 - bench[i, t]
        model += y[i, t] >= x[i, t] - bench[i, t]

# --- Captain Selection ---
for t in gameweeks:
    model += lpSum(c[i, t] for i in range(num_players)) == 1
    for i in range(num_players):
        model += c[i, t] <= y[i, t]

# --- Budget Constraints ---
# Update money in bank for t >= 1
for t in gameweeks[1:]:
    model += money_in_bank_var[t] == money_in_bank_var[t-1] + \
             lpSum(transfer_out[i, t] * list1[i] for i in range(num_players)) - \
             lpSum(transfer_in[i, t] * costs[i] for i in range(num_players))
# For each gameweek, squad value (using list1) plus money in bank equals available funds.
for t in gameweeks:
    model += lpSum(x[i, t] * list1[i] for i in range(num_players)) + money_in_bank_var[t] == budget_amount

# --- Maximum 3 Players per Team ---
for t in gameweeks:
    for team, indices in team_to_indices.items():
        model += lpSum(x[i, t] for i in indices) <= 3

# --- Transfer Constraints ---
for t in gameweeks[1:]:
    if t == wildcard_round:
        # Wildcard round: unlimited transfers
        for i in range(num_players):
            model += x[i, t] >= x[i, t-1] - transfer_out[i, t]
            model += x[i, t] <= x[i, t-1] + transfer_in[i, t]
    else:
        # Normal transfer constraints
        for i in range(num_players):
            model += transfer_in[i, t] >= x[i, t] - x[i, t-1]
            model += transfer_out[i, t] >= x[i, t-1] - x[i, t]
            model += transfer_out[i, t] <= x[i, t-1]
        model += lpSum(transfer_in[i, t] for i in range(num_players)) <= 1 + saved_transfers[t-1]

for t in gameweeks[1:]:
    if t == wildcard_round:
        model += saved_transfers[t] == 0  # Reset after wildcard
    else:
        model += saved_transfers[t] == saved_transfers[t-1] + (1 - lpSum(transfer_in[i, t] for i in range(num_players)))
        model += saved_transfers[t] <= 5

# --- Initial Transfers & Bank ---
model += saved_transfers[0] == initial_saved
model += money_in_bank_var[0] == money_in_bank_init

# --- Solve the Model ---
model.solve(PULP_CBC_CMD(msg=True, timeLimit=550))
#model.solve(PULP_CBC_CMD(msg=True))
# Display Results
print(f"Status: {model.status}")
print(f"\nTotal Optimized Predicted Points: {model.objective.value()}")
# Display Squad & Transfers
for t in range(1, optimize_range):
    print(f"\nGameweek {t+Last_GW} Squad:")
    for i in range(num_players):
        if x[i, t].varValue > 0.5:
            status = "Bench" if bench[i, t].varValue > 0.5 else "Playing"
            print(f"- {players[i]} ({positions[i]}) - {status}")

# Display Transfers for Each Gameweek
for t in range(1, optimize_range):  
    print(f"\nTransfers for Gameweek {t+Last_GW}:")
    if t == wildcard_round:
        print("  Wildcard Activated! All players can be changed.")
    else:
        players_in = [players[i] for i in range(num_players) if transfer_in[i, t].varValue > 0.5]
        players_out = [players[i] for i in range(num_players) if x[i, t-1].varValue > 0.5 and x[i, t].varValue < 0.5]

        if players_in or players_out:
            print(f"  In: {', '.join(players_in) if players_in else 'None'}")
            print(f"  Out: {', '.join(players_out) if players_out else 'None'}")
        else:
            print("  No transfers this week.")
# Display Captains
for t in range(1, optimize_range):
    for i in range(num_players):
        if c[i, t].varValue > 0.5:
            print(f"Gameweek {t+Last_GW} Captain: {players[i]}")



<>:96: SyntaxWarning: invalid escape sequence '\F'
<>:96: SyntaxWarning: invalid escape sequence '\F'
C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_32824\4051872944.py:96: SyntaxWarning: invalid escape sequence '\F'
  data=pd.read_csv("Raw_Data_24\Fantasy_season_2024_data.csv")


Max event: 28
1
    element_in  element_in_cost               Full_Name  element  value  \
0          447             72.0              Chris_Wood      447     72   
1          533             47.0         Rayan_Aït-Nouri      533     47   
3           74             51.0          Dango_Ouattara       74     52   
4          235             50.0         Jordan_Pickford      235     51   
5          580             44.0            Dean_Huijsen      580     45   
6          231             43.0       Vitalii_Mykolenko      231     44   
7          211             47.0            Daniel_Muñoz      211     50   
8          366             84.0  Bruno_Borges Fernandes      366     83   
9          110             62.0             Yoane_Wissa      110     65   
10         311             70.0  Trent_Alexander-Arnold      311     75   
11         401             85.0          Alexander_Isak      401     94   
12         328            131.0           Mohamed_Salah      328    138   
13       

In [ ]:
With hits

In [ ]:
import pandas as pd
from pulp import LpMaximize, LpProblem, LpVariable, lpSum, PULP_CBC_CMD
import requests

# --------------------------
# Load Data and Setup
# --------------------------
data = pd.read_csv("Optimize_players.csv")
squad = pd.read_csv("Squad_data.csv")

team_id = 544468  # Replace with your FPL team ID
url = f"https://fantasy.premierleague.com/api/entry/{team_id}/"
response = requests.get(url)
if response.status_code == 200:
    resonsep_data = response.json()
else:
    print(f"Error fetching data (Status Code: {response.status_code})")

money_in_bank_init = resonsep_data.get("last_deadline_bank", 0) / 10  # Convert to actual value

players = data['Name'].tolist()
costs = data['value'].tolist()

# Determine initial squad indices from your squad file
initial_squad = []
for t in range(len(squad)):
    name = squad["Full_Name"].values[t]
    initial_squad.append(players.index(name))

# Create a modified cost list for budget constraints:
# For players already owned, use their selling price (cheaper than buying cost)
list1 = costs.copy()
selling_cost = squad["selling_price"].values

# Calculate total available funds if you sold your squad
budget_amount = sum(selling_cost) + money_in_bank_init
print("Budget amount (if squad sold):", budget_amount)

# Update list1 for players in your initial squad to reflect selling price
for i in range(len(selling_cost)):
    list1[initial_squad[i]] = selling_cost[i]

# Other constants
budget = 103.0  # (Not used further if using budget_amount)
positions = data['position'].tolist()
teams = data['team'].tolist()
predicted_points = data[['p0', 'p1', 'p2', 'p3', 'p4', 'p5', 'p6', 'p7', 'p8']].values

# --------------------------
# Define Gameweeks and Parameters
# --------------------------
optimize_range = 9  # Number of gameweeks to optimize
gameweeks = range(optimize_range)
num_players = len(players)
wildcard_round = 3  # Gameweek 3 (index 2) is wildcard
# Note: bench_points_gw was set to 20 in your code but gameweeks indices run 0 to 8.
# Adjust if needed.
bench_points_gw = None  

# Hit penalty: points deducted per extra transfer (commonly 4 points)
hit_penalty = 4

# --------------------------
# Define the Model
# --------------------------
model = LpProblem("Maximize_Predicted_Points", LpMaximize)

# Decision Variables:
# x[i,t] = 1 if player i is in the squad for gameweek t
x = {(i, t): LpVariable(cat='Binary', name=f"x_{i}_{t}") 
     for i in range(num_players) for t in gameweeks}

# bench[i,t] = 1 if player i is on the bench for gameweek t
bench = {(i, t): LpVariable(cat='Binary', name=f"bench_{i}_{t}") 
         for i in range(num_players) for t in gameweeks}

# c[i,t] = 1 if player i is selected as captain for gameweek t
c = {(i, t): LpVariable(cat='Binary', name=f"captain_{i}_{t}") 
     for i in range(num_players) for t in gameweeks}

# y[i,t] = 1 if player i is in the starting XI (i.e. playing) for gameweek t
y = {(i, t): LpVariable(cat='Binary', name=f"y_{i}_{t}") 
     for i in range(num_players) for t in gameweeks}

# Transfer decision variables (1 if transfer in/out is made)
transfer_in = {(i, t): LpVariable(cat='Binary', name=f"transfer_in_{i}_{t}") 
               for i in range(num_players) for t in gameweeks}
transfer_out = {(i, t): LpVariable(cat='Binary', name=f"transfer_out_{i}_{t}") 
                for i in range(num_players) for t in gameweeks}

# Saved transfers (free transfers carried over); max 3 allowed.
saved_transfers = {t: LpVariable(cat='Integer', lowBound=0, upBound=3, name=f"saved_transfers_{t}") 
                   for t in gameweeks}

# Money in bank for each gameweek
money_in_bank_var = {t: LpVariable(f"money_in_bank_{t}", lowBound=0, cat='Continuous') 
                     for t in gameweeks}

# NEW: Extra transfers taken (beyond free transfers) for gameweeks > 0.
# (No extra transfers for gameweek 0 as that is your initial squad.)
extra_transfers = {t: LpVariable(f"extra_transfers_{t}", lowBound=0, cat='Integer')
                   for t in gameweeks if t > 0}

# --------------------------
# Initial Conditions & Squad Constraints
# --------------------------
# Force the initial squad in gameweek 0
for i in range(num_players):
    model += x[i, 0] == (1 if i in initial_squad else 0)

# --------------------------
# Objective Function
# --------------------------
# Sum predicted points for players in starting XI (and captain bonus) minus hit penalties.
# (If you want to include bench points, adjust accordingly.)
obj = lpSum((y[i, t] + c[i, t]) * predicted_points[i][t] 
            for i in range(num_players) for t in gameweeks)

# Subtract hit penalties for extra transfers (only for gameweeks > 0)
obj -= hit_penalty * lpSum(extra_transfers[t] for t in gameweeks if t > 0)

model += obj

# --------------------------
# Squad & Position Constraints
# --------------------------
for t in gameweeks:
    # Squad size must be 15
    model += lpSum(x[i, t] for i in range(num_players)) == 15
    
    # Position constraints (example numbers; adjust to your league rules)
    model += lpSum(x[i, t] for i in range(num_players) if positions[i] == 'DEF') == 5
    model += lpSum(x[i, t] for i in range(num_players) if positions[i] == 'GK') == 2
    model += lpSum(x[i, t] for i in range(num_players) if positions[i] == 'MID') == 5
    model += lpSum(x[i, t] for i in range(num_players) if positions[i] == 'FWD') == 3

    # Bench constraints: 1 GK and 3 outfield (if needed)
    model += lpSum(bench[i, t] for i in range(num_players) if positions[i] == 'GK') == 1
    model += lpSum(bench[i, t] for i in range(num_players) if positions[i] != 'GK') == 3
    for i in range(num_players):
        model += bench[i, t] <= x[i, t]

# --------------------------
# Starting XI Constraints
# --------------------------
for t in gameweeks:
    # Ensure at least 11 players are selected to play.
    # (You already have 15 in squad and bench split, but you can add an explicit constraint if desired)
    model += lpSum(y[i, t] for i in range(num_players)) == 11
    for i in range(num_players):
        # A player can only be in the starting XI if they are in the squad and not on the bench.
        model += y[i, t] <= x[i, t]
        model += y[i, t] <= 1 - bench[i, t]
        model += y[i, t] >= x[i, t] - bench[i, t]

# --------------------------
# Captain Constraints
# --------------------------
for t in gameweeks:
    # Exactly one captain per gameweek
    model += lpSum(c[i, t] for i in range(num_players)) == 1
    for i in range(num_players):
        model += c[i, t] <= y[i, t]

# --------------------------
# Money in Bank and Budget Constraints
# --------------------------
# Set initial money in bank
model += money_in_bank_var[0] == money_in_bank_init

# Update money in bank for subsequent gameweeks based on transfers:
for t in gameweeks[1:]:
    model += money_in_bank_var[t] == money_in_bank_var[t-1] + \
             lpSum(transfer_out[i, t] * list1[i] for i in range(num_players)) - \
             lpSum(transfer_in[i, t] * costs[i] for i in range(num_players))

# Enforce that for each gameweek, the squad’s total (using selling prices for owned players) plus money in bank does not exceed budget_amount.
for t in gameweeks:
    model += lpSum(x[i, t] * list1[i] for i in range(num_players)) + money_in_bank_var[t] <= budget_amount

# --------------------------
# Transfer and Extra Transfer Constraints
# --------------------------
# (Remove the original hard cap so extra transfers can be taken.)
for t in gameweeks[1:]:
    # For non-wildcard gameweeks, impose transfer consistency:
    if t == wildcard_round:
        # Wildcard round: allow unlimited transfers
        for i in range(num_players):
            model += x[i, t] >= x[i, t - 1] - transfer_out[i, t]
            model += x[i, t] <= x[i, t - 1] + transfer_in[i, t]
    else:
        # For normal gameweeks, allow transfers but account for extra transfers:
        for i in range(num_players):
            model += transfer_in[i, t] >= x[i, t] - x[i, t - 1]
            model += transfer_out[i, t] >= x[i, t - 1] - x[i, t]
            model += transfer_out[i, t] <= x[i, t - 1]
            
        # Instead of a hard cap, let free transfers be 1 + saved_transfers[t-1] and define extra transfers:
        total_transfers = lpSum(transfer_in[i, t] for i in range(num_players))
        free_transfer_allowance = 1 + saved_transfers[t-1]
        # extra_transfers[t] should capture any transfers above the free allowance:
        model += extra_transfers[t] >= total_transfers - free_transfer_allowance
        # (The solver will set extra_transfers[t] to the minimum value satisfying this constraint.)
        
# Update saved transfers (free transfers carry over only if not used, but are lost if extra transfers are taken)
for t in gameweeks[1:]:
    # If you use fewer free transfers than available, you carry them over (up to a maximum of 3).
    # This constraint forces saved_transfers[t] to be at most what remains.
    model += saved_transfers[t] <= saved_transfers[t-1] + 1 - lpSum(transfer_in[i, t] for i in range(num_players))
    # Because saved_transfers are defined with a low bound of 0, if transfers exceed the free allowance, saved_transfers[t] will be 0.
    
# For gameweek 0, set saved_transfers to 0.
model += saved_transfers[0] == 0

# --------------------------
# Solve the Model
# --------------------------
model.solve()  # You can use PULP_CBC_CMD(msg=True) for verbose output

# --------------------------
# Display Results
# --------------------------
print(f"Status: {model.status}")
print(f"Total Optimized Predicted Points: {model.objective.value()}")

for t in range(1, optimize_range):
    print(f"\nGameweek {t+27} Squad:")
    for i in range(num_players):
        if x[i, t].varValue > 0.5:
            status = "Bench" if bench[i, t].varValue > 0.5 else "Playing"
            print(f"- {players[i]} ({positions[i]}) - {status}")

for t in range(1, optimize_range):
    print(f"\nTransfers for Gameweek {t+28}:")
    if t == wildcard_round:
        print("  Wildcard Activated! All players can be changed.")
    else:
        players_in = [players[i] for i in range(num_players) if transfer_in[i, t].varValue > 0.5]
        players_out = [players[i] for i in range(num_players) if x[i, t-1].varValue > 0.5 and x[i, t].varValue < 0.5]
        extra = extra_transfers[t].varValue
        if players_in or players_out:
            print(f"  In: {', '.join(players_in) if players_in else 'None'}")
            print(f"  Out: {', '.join(players_out) if players_out else 'None'}")
            print(f"  Extra Transfers Taken (Hit): {extra}")
        else:
            print("  No transfers this week.")

for t in range(1, optimize_range):
    for i in range(num_players):
        if c[i, t].varValue > 0.5:
            print(f"Gameweek {t+28} Captain: {players[i]}")
